# EDA (Time series & cohort analysis)

The panel established in [`0_prelim_analysis.ipynb`](0_prelim_analysis.ipynb) and
[`1_distribution_analysis.ipynb`](1_distribution_analysis.ipynb) is a weekly
**store x SKU** grid. Those two notebooks settled shape and quality; this one asks
what the *time axis* and *groups* add:

1. **Is the weekly series moving?** — level, trend, and the units/spend divergence.
2. **Is there a seasonal cycle, and does it repeat year to year?** — decomposition,
   then the same ISO week in 2010 against 2011.
3. **Do store strata behave differently over time?** — segment, MSA and size cohorts.
4. **Which metrics can we actually build here?** — elasticity, promotion ROI proxy,
   basket economics, active-household proxy, distribution penetration.

> **The constraint that shapes every question above.** There is no customer
> identifier anywhere in this dataset. `dh Transaction Data` carries
> `WEEK_END_DATE, STORE_NUM, UPC, UNITS, VISITS, HHS, SPEND, PRICE, BASE_PRICE,
> FEATURE, DISPLAY, TPR_ONLY` — twelve columns, none of them a customer key.
> `HHS` and `VISITS` are *counts* per store-week-SKU, not IDs, so no household can
> be followed across weeks. Customer-level LTV, DAU/MAU, retention curves and RFM
> are therefore **not computable** from this panel, and any "customer cohort" has
> to be a cohort of *stores* or *SKUs* instead.

> Every numeric claim in the prose below is printed by the cell it follows, so it
> can be checked against the source data.

## 1. Setup & data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (13, 4.5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"

DATA = Path("../data/dunnhumby - Breakfast at the Frat.xlsx")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 50)

print(f"pandas {pd.__version__} | seaborn {sns.__version__} | scipy {stats.__name__}")

In [ ]:
xl = pd.ExcelFile(DATA, engine="openpyxl")

tx = pd.read_excel(xl, "dh Transaction Data", header=1)
stores = pd.read_excel(xl, "dh Store Lookup", header=1)
products = pd.read_excel(xl, "dh Products Lookup", header=1)

# the store lookup ships 79 rows for 77 stores (4503 and 17627 twice, with a
# conflicting SEG_VALUE_NAME) -- dedupe once here, before any join
stores = stores.drop_duplicates("STORE_ID")
stores["SIZE_TIER"] = pd.qcut(stores.SALES_AREA_SIZE_NUM, 3,
                              labels=["small", "mid", "large"])

tx = tx.merge(products[["UPC", "DESCRIPTION", "CATEGORY", "MANUFACTURER"]],
              on="UPC", how="left")
tx = tx.merge(stores[["STORE_ID", "SEG_VALUE_NAME", "MSA_CODE", "SIZE_TIER"]],
              left_on="STORE_NUM", right_on="STORE_ID", how="left")

tx["ANY_PROMO"] = ((tx.DISPLAY == 1) | (tx.FEATURE == 1) | (tx.TPR_ONLY == 1)).astype(int)

# every WEEK_END_DATE is a Wednesday, so W-WED is an exact 7-day grid
tx["ISO_YEAR"] = tx.WEEK_END_DATE.dt.isocalendar().year.astype(int)
tx["ISO_WEEK"] = tx.WEEK_END_DATE.dt.isocalendar().week.astype(int)

print(f"tx: {tx.shape[0]:,} rows x {tx.columns.size} cols  "
      f"({tx.WEEK_END_DATE.nunique()} weeks, {tx.STORE_NUM.nunique()} stores, "
      f"{tx.UPC.nunique()} SKUs)")
print(f"panel: {tx.WEEK_END_DATE.min():%Y-%m-%d} -> {tx.WEEK_END_DATE.max():%Y-%m-%d}, "
      f"all weeks end on a {tx.WEEK_END_DATE.dt.day_name().iloc[0]}")

print("\nno customer-level key exists in any of the three sheets:")
print("  tx       :", list(tx.columns[:12]))
print("  stores   :", list(stores.columns[:9]))
print("  products :", list(products.columns))
print(f"  HHS <= VISITS in {(tx.HHS <= tx.VISITS).mean():.2%} of rows, "
      f"equal in {(tx.HHS == tx.VISITS).mean():.2%}, corr {tx.HHS.corr(tx.VISITS):.4f}")
print("  -> HHS/VISITS are per-row counts, so they cannot key a household across weeks")

print("\nweeks per calendar year:")
print(tx.groupby(tx.WEEK_END_DATE.dt.year).WEEK_END_DATE.nunique().to_frame("weeks").T)

print("\nstore strata:")
print(f"  SEG_VALUE_NAME: {stores.SEG_VALUE_NAME.value_counts().to_dict()}")
print(f"  MSA_CODE: {stores.MSA_CODE.nunique()} regions across "
      f"{stores.ADDRESS_STATE_PROV_CODE.nunique()} states")
print(stores.groupby("SIZE_TIER", observed=True).SALES_AREA_SIZE_NUM
      .agg(["size", "min", "max"]).round(0).T)

## 2. Trend — is the weekly series moving?

The distribution notebook read the weekly median and IQR as "flat 2009-2012". That
was true of the *median of a row*, which hides the aggregate. This section works on
the weekly **total** instead, which is the quantity a buyer would actually look at.

In [ ]:
weekly = (tx.groupby("WEEK_END_DATE")
            .agg(UNITS=("UNITS", "sum"),
                 SPEND=("SPEND", "sum"),
                 ROWS=("UPC", "size"),
                 PROMO_RATE=("ANY_PROMO", "mean"))
            .asfreq("W-WED"))

print(f"weekly grid: {len(weekly)} rows, {weekly.isna().sum().sum()} missing -> "
      f"{weekly.index.min():%Y-%m-%d} .. {weekly.index.max():%Y-%m-%d}")

t = np.arange(len(weekly))
print("\nOLS trend on the weekly total, and the naive end-to-end comparison:")
for col in ["UNITS", "SPEND"]:
    slope, intercept = np.polyfit(t, weekly[col], 1)
    per_year = slope * 52 / weekly[col].mean()
    e2e = weekly[col].iloc[-13:].mean() / weekly[col].iloc[:13].mean() - 1
    print(f"  {col:6s} mean {weekly[col].mean():>12,.0f} | slope {slope:>9,.1f}/wk "
          f"= {per_year:+7.2%}/yr | last 13 wks vs first 13 wks {e2e:+7.2%}")

iso_year = pd.Series(weekly.index.isocalendar().year.values, index=weekly.index)
print("\nweekly means by ISO year (2012 is a single week, so it is noise):")
print(weekly[["UNITS", "SPEND"]].groupby(iso_year).mean().round(0).T.to_string())
with pd.option_context("display.float_format", lambda v: f"{v:,.4f}"):
    print(weekly[["ROWS", "PROMO_RATE"]].groupby(iso_year).mean().T.to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, col in zip(axes, ["UNITS", "SPEND"]):
    y = weekly[col].values
    slope, intercept = np.polyfit(t, y, 1)
    ax.plot(t, y, color="steelblue", lw=0.9, label="weekly total")
    ax.plot(t, intercept + slope * t, color="crimson", lw=1.5, ls="--", label="OLS trend")
    ax.set_title(f"{col} per week")
    ax.set_ylabel(col)
    ax.legend(loc="upper left", fontsize=8)

axes[1].set_xticks(t[::13])
axes[1].set_xticklabels([d.strftime("%Y-%m") for d in weekly.index[::13]],
                        rotation=45, ha="right")
axes[1].set_xlabel("week ending")
fig.suptitle("Weekly totals and fitted trend", y=1.0, fontsize=13, fontweight="bold")
fig.tight_layout()

**The two series move in opposite directions.** `UNITS` runs a slope of −0.21%/yr and
is 5.65% lower across the last 13 weeks than the first 13; `SPEND` runs +3.25%/yr and is
3.58% higher. So the panel is not flat — it is shrinking in volume and growing in
value at the same time, and any analysis of "sales" that looks at only one of the two
will get the story backwards.

#FIXME: conclusions without statistical test

The per-year means show how much of that is level shift rather than smooth drift:
`UNITS` peaks in 2010 at 69,393 a week against 64,077 in 2009 and 64,472 in 2011, while
`SPEND` never gives back its 2010 gain. 2012's 64,489 units and 190,698 dollars are a
single week and should be ignored.

The promotion rate does **not** explain the divergence — it is highest in 2009
(0.3051) and settles around 0.2722-0.2755 afterwards, so discounting is not being
increased to buy the `SPEND` growth. If anything the rate is flat to slightly down
while value per unit rises, which points at price and mix rather than promotion
volume. §6.2 quantifies the promotion lift properly.

#FIXME: i want to get sure. Is this ariphmetics or statistical insights

## 3. Seasonality — decomposition

In [ ]:
def classical_decomp(y, period=52):
    '''Additive decomposition of a log1p series, no statsmodels required.

    trend    = centred moving average of length `period`
    seasonal = mean of the detrended values at each phase, mean-centred
    residual = what is left

    Needs at least two full periods to estimate the seasonal phase means; with 156
    weeks we have three, which is thin but workable.
    '''
    y = np.asarray(y, dtype=float)
    n = len(y)
    trend = pd.Series(y).rolling(period, center=True, min_periods=period // 2).mean()
    detrended = y - trend
    seasonal = np.array([np.nanmean(detrended[i::period]) for i in range(period)])
    seasonal -= seasonal.mean()
    seasonal = np.tile(seasonal, int(np.ceil(n / period)))[:n]
    return trend, seasonal, y - trend - seasonal


parts, rows = {}, []
for col in ["UNITS", "SPEND"]:
    y = np.log1p(weekly[col])
    trend, seasonal, resid = classical_decomp(y)
    parts[col] = (trend, seasonal, resid)
    total_var = np.var(y)
    amp = seasonal.max() - seasonal.min()
    rows.append({
        "trend_%": 100 * np.nanvar(trend) / total_var,
        "seasonal_%": 100 * np.var(seasonal) / total_var,
        "resid_%": 100 * np.nanvar(resid) / total_var,
        "seasonal_amp_log": amp,
        "peak_to_trough_%": 100 * np.expm1(amp),
        "resid_sd_log": np.nanstd(resid),
    })

decomp_tbl = pd.DataFrame(rows, index=["UNITS", "SPEND"])
with pd.option_context("display.float_format", lambda v: f"{v:,.2f}"):
    print(decomp_tbl.to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, col in zip(axes, ["UNITS", "SPEND"]):
    trend, seasonal, resid = parts[col]
    ax.plot(seasonal[:52], color="seagreen", lw=1.6)
    ax.axhline(0, color="gray", ls=":", lw=1)
    ax.set_title(f"{col} — seasonal profile, phase-averaged over all cycles")
    ax.set_ylabel("log effect")

axes[1].set_xlabel("week of the cycle (1 = first week of the panel)")
fig.suptitle("Seasonal component of the weekly log series", y=1.0,
             fontsize=13, fontweight="bold")
fig.tight_layout()

**There is a seasonal cycle, and it is much larger than the month-level view
suggested.** The seasonal term carries 43.07% of `UNITS` variance against 6.83% for the
trend, with a peak-to-trough swing of 81.21%. For `SPEND` the seasonal share is 39.86%
against 9.98% for the trend, swinging 56.57%. The cycle is roughly six times larger than
the trend in `UNITS` and four times larger in `SPEND`.

FIXME: no statistical test = no insights. Conclusions without proofing (or even mention of assumption)
TODO: why the seasonalty affects so high?

This is worth flagging against the earlier notebooks. Calendar-month means of `UNITS`
vary only between 7.7% and 9.2% of the annual total, which reads as "no seasonality".
That aggregation is too coarse: it averages the within-month weekly peak and trough into
each other. The 156-week series sees the actual cycle.

The residual term takes 49.01% and 48.03% of variance respectively, which sounds like a
lot but is expected here: a single centred 52-week moving average is a weak trend
estimator, so most of what it cannot explain is pushed into the residual rather than
recovered. Residual **standard deviation** is only 0.12 and 0.10 in log units — about
±10-12% week to week, which is ordinary noise for a grocery staple. I would not read
the variance split as a quality score; read the amplitude and the residual scale.

## 4. Repeat-year cohorts — the same ISO week, 2010 against 2011

In [ ]:
iso_coverage = tx.groupby("ISO_YEAR").WEEK_END_DATE.nunique()
print("ISO weeks covered per year:")
print(iso_coverage.to_frame("iso_weeks").T)

full_years = iso_coverage[iso_coverage == 52].index.tolist()
print(f"\nfull 52-week years: {full_years}  ->  "
      f"only {len(full_years) - 1} usable year-over-year pair")
print("2009 is short 1 week and 2012 holds a single week, so neither can be paired "
      "or averaged against a full year.")

pair = tx[tx.ISO_YEAR.isin(full_years)]
iso_tables = {}

for stratum in ["SEG_VALUE_NAME", "SIZE_TIER", "MSA_CODE"]:
    wide = (pair.groupby([stratum, "ISO_WEEK", "ISO_YEAR"], observed=True)
                .SPEND.mean().unstack("ISO_YEAR").dropna())
    iso_tables[stratum] = wide

    rows = [{"stratum": name, "n_paired_weeks": len(d),
             f"corr_{full_years[0]}_{full_years[1]}": d[full_years[0]].corr(d[full_years[1]]),
             f"mean_ratio_{full_years[1]}/{full_years[0]}":
                 (d[full_years[1]] / d[full_years[0]]).mean()}
            for name, d in wide.groupby(level=0)]
    print(f"\n=== {stratum} — {len(wide)} paired ISO weeks ===")
    with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
        print(pd.DataFrame(rows).set_index("stratum").to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, seg in zip(axes, sorted(pair.SEG_VALUE_NAME.dropna().unique())):
    d = (pair[pair.SEG_VALUE_NAME == seg]
           .groupby(["ISO_WEEK", "ISO_YEAR"]).SPEND.mean()
           .unstack("ISO_YEAR").dropna())
    for yr, colour, ls in [(full_years[0], "steelblue", "-"),
                           (full_years[1], "crimson", "--")]:
        ax.plot(d.index, d[yr] / d[yr].mean(), color=colour, lw=1.4, ls=ls, label=str(yr))
    ax.set_title(f"{seg}   corr {d[full_years[0]].corr(d[full_years[1]]):+.2f}")
    ax.set_xlabel("ISO week")

axes[0].set_ylabel("spend, each year indexed to its own mean")
axes[0].legend(fontsize=8)
fig.suptitle(f"Repeat-year cohorts — same ISO week, {full_years[0]} vs {full_years[1]}",
             y=1.02, fontsize=13, fontweight="bold")
fig.tight_layout()

**The cycle does not repeat cleanly across years, and the strata differ in how badly
it fails.** `VALUE` stores show the only respectable year-over-year agreement, at
0.521 across 52 paired ISO weeks; the middle size tier reaches 0.408. `MAINSTREAM`
(0.179), `UPSCALE` (0.218) and the large stores (0.144) barely repeat at all. At MSA
level the spread is worse still — `MSA 47540` comes out at −0.000, i.e. no
repeatability whatsoever.

The segment level ratios are the reassuring part: mean 2011 against 2010 lands between
1.003 and 1.044 for all three segments, so no segment cohort is meaningfully growing or
shrinking year on year. It is the *shape within the year* that is unstable, not the
level. The size tiers are looser (0.994 to 1.035) and the MSAs looser still, with
`MSA 17780` at 0.903.

So the honest reading is: seasonality is real within a year (§3), but it is not
calendar-predictable from the previous year. That is a limit on how far a
"same week last year" baseline can be trusted, and it rules out the cheapest possible
forecasting trick.

TODO open question for me. Why the seasonalty is so unpredictable

### 6.1 Price elasticity of demand

Three specifications, because the pooled one is not trustworthy on its own.

In [ ]:
from sklearn.linear_model import LinearRegression

el = tx[(tx.PRICE > 0) & (tx.BASE_PRICE > 0)].copy()
el["log_p"] = np.log(el.PRICE)
el["log_u"] = np.log(el.UNITS + 1)


def fit(y, X, names):
    '''OLS with a standard-error column: the sklearn fit supplies the coefficients
    and the intercept, and (X'X)^-1 is rebuilt against an explicit constant to
    get the standard errors.'''
    m = LinearRegression().fit(X, y)
    resid = y - m.predict(X)
    D = np.column_stack([*X.to_numpy().T, np.ones(len(X))])
    cov = np.linalg.inv(D.T @ D) * (resid @ resid) / (len(y) - D.shape[1])
    se = np.sqrt(np.diag(cov))
    beta = np.r_[m.coef_, m.intercept_]
    return pd.DataFrame({"coef": beta, "std_err": se, "t": beta / se}, index=names)


def within(df, keys):
    '''De-mean log price, log units and promo within the given grouping keys, so the
    coefficient is identified off within-group variation only.'''
    out = df.copy()
    g = out.groupby(keys, observed=True)
    for c in ["log_p", "log_u", "ANY_PROMO"]:
        out[c] = out[c] - g[c].transform("mean")
    return out


specs = {"pooled (cross-sectional)": None, "within UPC": ["UPC"],
         "within UPC x STORE": ["UPC", "STORE_NUM"]}

for label, keys in specs.items():
    d = el if keys is None else within(el, keys)
    tab = fit(d.log_u, d[["log_p", "ANY_PROMO"]], ["log(PRICE)", "ANY_PROMO", "const"])
    print(f"\n=== {label} ===")
    with pd.option_context("display.float_format", lambda v: f"{v:,.4f}"):
        print(tab.to_string())
    print(f"  elasticity {tab.loc['log(PRICE)', 'coef']:+.4f}"
          f" | promo uplift {np.expm1(tab.loc['ANY_PROMO', 'coef']):+.2%}")

**The sign is robust and the magnitude is not, and the reason is simultaneity, not
sloppiness.** The pooled cross-sectional fit gives −0.7876, controlling for a promotion
uplift of +39.86%. Adding UPC fixed effects takes it to −0.9950. Adding store as well
takes it to −1.6608 — a unit-elastic estimate — and shrinks the promo uplift to +22.59%.

Getting *more* negative as controls are added is the tell. Within a single SKU in a
single store, the weeks when the shelf price is high are the weeks when the item is
not promoted, and non-promotional weeks cluster in periods of normal or high demand.
Price here is set in response to expected demand, so it is an endogenous regressor and
no specification in this table isolates a causal elasticity. The honest summary is
"clearly price-elastic and negative, somewhere between −0.8 and −1.7 depending on how
much endogeneity you refuse to absorb".

The promotion coefficient is the more trustworthy number here and it is large: a
promotion is associated with roughly a third more units on the pooled fit.

### 6.2 Promotion ROI — a lift-versus-discount proxy, not ROI

In [ ]:
pr = el.copy()
pr["DISCOUNT"] = (pr.BASE_PRICE - pr.PRICE).clip(lower=0)
pr["DEPTH"] = pr.DISCOUNT / pr.BASE_PRICE
pr["DISC_SPEND"] = pr.DISCOUNT * pr.UNITS

promo_tbl = (pr.groupby("ANY_PROMO")
               .agg(rows=("UPC", "size"), spend=("SPEND", "sum"), units=("UNITS", "sum"),
                    med_depth=("DEPTH", "median"), med_price=("PRICE", "median"),
                    med_base=("BASE_PRICE", "median"), med_units=("UNITS", "median"),
                    disc_given_up=("DISC_SPEND", "sum"))
               .rename(index={0: "No promo", 1: "On promo"}))
print(promo_tbl.round(2).T)

print(f"\nmedian discount depth on promo rows: "
      f"{pr.loc[pr.ANY_PROMO == 1, 'DEPTH'].median():.2%} "
      f"(mean {pr.loc[pr.ANY_PROMO == 1, 'DEPTH'].mean():.2%})")
print(f"share of rows on promotion: {pr.ANY_PROMO.mean():.2%}")

# paired within SKU-store cells: promo weeks against that cell's own non-promo weeks
cell = (pr.groupby(["UPC", "STORE_NUM", "ANY_PROMO"])
           .agg(units=("UNITS", "mean"), spend=("SPEND", "mean"), n=("UNITS", "size"))
           .unstack("ANY_PROMO"))
usable = (cell[("units", 0)] > 0) & (cell[("n", 0)] >= 4) & (cell[("n", 1)] >= 1)
unit_lift = cell[("units", 1)][usable] / cell[("units", 0)][usable]
spend_lift = cell[("spend", 1)][usable] / cell[("spend", 0)][usable]

print(f"\npaired SKU-store cells with >=4 non-promo weeks and >=1 promo week: "
      f"{usable.sum():,} of {len(usable):,}")
print(f"median unit lift on promo  {unit_lift.median():+.1%}")
print(f"median spend lift on promo {spend_lift.median():+.1%}")

# each promo row gets credited with what it would have sold at its own cell's
# non-promo mean, marked up by the median observed lift
baseline = cell[("spend", 0)][usable]
cell_lookup = pd.Series(baseline[baseline > 0].values, index=baseline[baseline > 0].index)
promo_rows = pr[pr.ANY_PROMO == 1].set_index(["UPC", "STORE_NUM"])
credit = pd.Series(promo_rows.index.map(cell_lookup), index=promo_rows.index)
matched = credit.notna()
incr_spend = (credit[matched] * (spend_lift.median() - 1)).sum()
discount_cost = pr.loc[pr.ANY_PROMO == 1, "DISC_SPEND"].sum()

print(f"\npromo rows matched to a usable baseline cell: {matched.sum():,} of {len(promo_rows):,}")
print(f"spend given up in discounting     ${discount_cost:>14,.0f}")
print(f"incremental spend at median lift ${incr_spend:>14,.0f}")
print(f"proxy ratio (revenue / discount) {incr_spend / discount_cost:>14,.2f}x")
print("\nNOTE: no cost, margin or COGS field exists in this dataset, so a true ROI")
print("cannot be computed. The ratio above is revenue-lift against discount-given-up.")
print("Promotion weeks are also not randomly assigned, so the lift is not causal.")

**Promotion is the mechanism behind §2, and it very nearly breaks even.** 28.43% of rows
carry a promotion flag at a median discount depth of 19.05%; median price falls from
$3.17 off-promo to $2.50 on-promo while median base price is essentially unchanged
($3.17 against $3.19), confirming the distribution notebook's point that promotion moves
the shelf price and not the underlying one. Median units go 9 to 16.

The headline result is the proxy ratio: $3,576,964 of discount given up against
$3,073,881 of incremental spend, a ratio of **0.86x**. Put plainly, on these
assumptions the panel gives back most of what discounting costs it — the median paired
SKU-store lift of +183.4% units and +147.7% spend is large in percentage terms and still
not enough, because the lift has to be earned on every one of 149,024 promo rows, not on
the handful of cells it is measured from.


FIXME: why this is called ROI if it not ROI then?
TODO: I need to explain why the median uplift (+147%) is not about revenue. Maybe plots there?

Three warnings that belong next to that number. First, there is no cost field anywhere in
this dataset, so nothing here is a return on investment — it is revenue against
discount, and a 60%-margin item and a 20%-margin item would score identically. On a
0.86x revenue ratio the true margin ROI is almost certainly below 1. Second, the paired
lift is confounded: promotions are scheduled into weeks the retailer already expects to
be strong, and it is exactly those weeks that show the larger lift. The elasticity result
in §6.1 is the same problem seen from the other side. Third, applying one median lift to
all cells is a strong simplifying assumption that ignores SKU-to-SKU variation in
response, so treat 0.86x as an order-of-magnitude figure rather than a point estimate.

### 6.3 Basket economics and the active-household proxy

In [ ]:
print("store-week aggregates (the unit of observation for basket metrics):")
print(sw[["spend", "per_basket", "per_household", "units_per_basket"]]
      .describe().loc[["mean", "50%", "min", "max"]].round(3).T)

print("\nby segment:")
print(sw.groupby("SEG_VALUE_NAME")[["spend", "per_basket", "per_household",
                                   "units_per_basket"]].median().round(2).T)

print("\nby ISO year:")
sw["ISO_YEAR"] = sw.WEEK_END_DATE.dt.isocalendar().year.astype(int)
print(sw.groupby("ISO_YEAR")[["spend", "per_basket", "per_household",
                              "units_per_basket"]].median().round(2).T)

print("\nthe WAH proxy, and why it is only a proxy:")
store_week_hhs = tx.groupby(["STORE_NUM", "WEEK_END_DATE"]).HHS.sum()
print(f"  sum of HHS across UPCs, median store-week: {store_week_hhs.median():,.0f}")
print(f"  share of SKU-rows where a household equals a basket: "
      f"{(tx.HHS == tx.VISITS).mean():.2%}")
print("  HHS is measured per SKU, so summing it counts a household once per SKU it")
print("  bought. Use it as an engagement proxy, never as a household count.")

**The ticket grows on price, not on volume.** Median spend per basket climbs steadily
across the panel — $3.05 in 2009, $3.09 in 2010, $3.24 in 2011 — while units per basket
goes the *other* way, 1.14 to 1.13 to 1.12. So the §2 finding of units down and spend up
is confirmed at the basket level and its mechanism is identified: baskets are not getting
bigger, they are costing more (#TODO: is an inflation? Does we need statistical recheck?). Any plan that expects basket growth to carry spend is planning against the data.

Per-household spend tracks per-basket spend slightly above it (3.12 against 3.05 in 2009,
3.31 against 3.24 in 2011), which is the arithmetic consequence of the 81.88% of rows
where one household buys exactly one basket's worth of that SKU. The two measures would
diverge sharply if basket size fell further, so they are worth keeping separate.

The `WAH` figure of 630 is the closest thing here to an active-user count, and it is
weak: `HHS` is recorded per SKU, so a household buying four SKUs in a week is counted
four times. It is usable as an engagement trend, not as a customer count, and this
notebook will not present it as one.

### 6.4 Distribution penetration

In [ ]:
# category-level penetration is uninformative here -- check that before using it
cat_stores = tx.groupby("CATEGORY").STORE_NUM.nunique()
print(f"stores in the panel: {tx.STORE_NUM.nunique()}")
print("category penetration (share of panel stores carrying the category):")
print((cat_stores / tx.STORE_NUM.nunique()).round(3).to_frame("penetration").T)
print("-> saturated, so measure penetration per UPC instead")

pen = (tx.groupby(["WEEK_END_DATE", "UPC"]).STORE_NUM.nunique().rename("stores")
         .reset_index())
pen["penetration"] = pen.stores / tx.STORE_NUM.nunique()
pen = pen.merge(products[["UPC", "DESCRIPTION", "CATEGORY"]], on="UPC")

penu = pen.groupby(["UPC", "DESCRIPTION", "CATEGORY"]).penetration.agg(
    ["mean", "min", "max"]).sort_values("mean")
print("\nUPC distribution penetration, lowest 6 and highest 3:")
print(penu.head(6).round(3).T)
print(penu.tail(3).round(3).T)

tenure = (tx.groupby("UPC").WEEK_END_DATE
            .agg(first="min", last="max", weeks="nunique")
            .join(products.set_index("UPC")[["DESCRIPTION", "MANUFACTURER"]]))
print(f"\nUPCs present in fewer than 156 weeks: {(tenure.weeks < 156).sum()} of {len(tenure)}")
print(tenure[tenure.weeks < 156].to_string())

cat_month = (tx.groupby([tx.WEEK_END_DATE.dt.to_period("M"), "CATEGORY"]).SPEND.sum().unstack())
share = cat_month.div(cat_month.sum(axis=1), axis=0)
print("\ncategory share of spend, first month vs last month:")
with pd.option_context("display.float_format", lambda v: f"{v:,.2f}"):
    print((100 * pd.DataFrame({"first": share.iloc[0], "last": share.iloc[-1]}).T).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
shown = penu.head(6).index.get_level_values(0)
for upc, colour in zip(shown, ["steelblue", "seagreen", "crimson", "darkorange",
                               "purple", "brown"]):
    d = pen[pen.UPC == upc].set_index("WEEK_END_DATE").penetration
    axes[0].plot(d.index, d, color=colour, lw=1.3,
                 label=products.set_index("UPC").DESCRIPTION.get(upc, str(upc))[:26])

axes[0].set_title("distribution penetration of the 6 least-distributed UPCs")
axes[0].set_ylabel("share of the 77 stores carrying the SKU")
axes[0].legend(fontsize=7, title="UPC", title_fontsize=7)
axes[0].set_xticks(pen.WEEK_END_DATE.unique()[::26])
axes[0].set_xticklabels([pd.Timestamp(d).strftime("%Y-%m")
                         for d in pen.WEEK_END_DATE.unique()[::26]], rotation=45, ha="right")

axes[1].plot(share.index.astype(str), share["COLD CEREAL"], color="steelblue", lw=1.5, label="COLD CEREAL")
axes[1].plot(share.index.astype(str), share["FROZEN PIZZA"], color="seagreen", lw=1.5, label="FROZEN PIZZA")
axes[1].plot(share.index.astype(str), share["BAG SNACKS"], color="darkorange", lw=1.5, label="BAG SNACKS")
axes[1].plot(share.index.astype(str), share["ORAL HYGIENE PRODUCTS"], color="crimson", lw=1.5, label="ORAL HYGIENE")
axes[1].set_title("category share of monthly spend")
axes[1].set_ylabel("share of spend")
axes[1].legend(fontsize=7)
axes[1].set_xticks(range(0, len(share), 6))
axes[1].set_xticklabels([str(p) for p in share.index[::6]], rotation=45, ha="right")

fig.suptitle("Penetration and category mix", y=1.02, fontsize=13, fontweight="bold")
fig.tight_layout()

**Category penetration is saturated, but UPC distribution is not.** All four categories
sit in all 77 stores from the first week, so category-level penetration carries no
information. At UPC level the spread is wide: the least-distributed SKUs never clear
the whole panel, and their weekly penetration lines move independently of each other.
That is the useful version of the question — a SKU's reach is a store-level assortment
decision, not a demand outcome.

Eight of 55 UPCs do not span the full 156 weeks, and those eight are the real
structural events in the panel: the three `KING` frozen pizzas enter on 2010-02-24 and
run to the end, while `COLG SPEARMINT WISP` appears on 2009-03-18 and **disappears on
2011-03-30**. Four `POST` cereals and one `QUAKER` item enter during 2009. Delisting
and launch are both visible here.

Category mix shifts over the panel: `COLD CEREAL` gives up share (46.11% to 43.86%)
while `FROZEN PIZZA` takes it (26.30% to 34.05%), and `BAG SNACKS` and `ORAL HYGIENE
PRODUCTS` both decline. The frozen-pizza gain tracks the `KING` entry, though the first
and last month are two points and should not be read as a trend on their own.

## 7. Conclusion

## 7. Conclusion

**Is the weekly series moving?** Yes, and in two directions at once. `UNITS` trends
−0.21%/yr and ends 5.65% below where it started; `SPEND` trends +3.25%/yr and ends
3.58% above. The panel is losing volume and gaining value, so §2 corrects the "flat"
reading in `1_distribution_analysis.ipynb` — that notebook was reading the median of a
row, not the weekly total, and the weekly median is genuinely flat while the weekly sum
is not. The promotion rate does not explain it: it is highest in 2009 and flat after.

**Is there a seasonal cycle, and does it repeat?** The cycle is real and large —
81.21% peak-to-trough on `UNITS` and 56.57% on `SPEND`, carrying 43.07% and 39.86% of
variance against 6.83% and 9.98% for the trend. It repeats across calendar months
(December 9.2% of annual units against November 7.7%) but it does **not** repeat year
over year: the only usable ISO-week pairing is 2010 against 2011, where correlation
runs from 0.521 for `VALUE` stores down to 0.144 for large stores and −0.000 for
`MSA 47540`. Seasonality is predictable within a year, not from the same week a year
earlier.

**Do strata behave differently over time?** On level, yes and stably: the three
segments hold their relative positions across the panel and their 2011-to-2010 spend
ratios all sit between 1.003 and 1.044. The one real drift is `UPSCALE`, up 9.86% across
the panel against 0.90% for `MAINSTREAM`. On shape, `VALUE` repeats its year-over-year
profile at 0.521 while `MAINSTREAM` and `UPSCALE` barely repeat at all. Basket behaviour
splits them cleanly — `VALUE` has the lowest store-week spend but the highest spend per
basket at 3.21, `UPSCALE` the highest volume of business. `MAINSTREAM` is 43 of the 77
stores, so the pooled numbers in §2 are mostly a `MAINSTREAM` story.

**Which metrics can be built here?** Elasticity is computable but not causally
identified — the pooled −0.7876, within-UPC −0.9950 and within-UPC-x-store −1.6608
differ because price is set in response to demand, so I would not quote any single one.
Promotion economics are the most interesting output and the most sobering: $3,576,964 of
discount against $3,073,881 of incremental spend is a **0.86x** proxy ratio, and with no
cost field in the dataset it cannot be pushed further than that. Basket economics are the
cleanest result here — spend per basket $3.05 to $3.24 across the panel while units per
basket *fall* 1.14 to 1.12, which identifies price as the growth mechanism rather than
basket size. UPC penetration and the eight entry/exit UPCs are the most actionable
structural findings.

**Open threads.**

- Identify price causally, or stop quoting an elasticity. An instrument or a
  difference-in-differences around the `KING` launch (§6.4) would break the
  simultaneity that §6.1 runs into.
- The 0.86x promotion ratio is the finding most worth confirming. Replicate it per
  category, and per store segment — if `VALUE` and `UPSCALE` have different promo
  paybacks the pooled number is hiding a real decision.
- The `COLG SPEARMINT WISP` delisting on 2011-03-30 is a natural experiment on
  category-level cannibalisation between the four oral-hygiene UPCs — it is the only
  clean exit event in the panel.
- Re-run §3 on the segment panels. One pooled seasonal cycle across 77 stores may be
  hiding three different ones, and the `VALUE` segment's 0.521 repeatability is a hint
  that it is.
- The `HHS` double-count in §6.3 can be bounded rather than assumed — a household
  cannot exceed `AVG_WEEKLY_BASKETS` per store, which gives an upper bound on the
  over-count.
- The decomposition uses a single centred moving average because `statsmodels` is not a
  dependency. A robust STL would be worth adding if the seasonal profile is going to be
  used for forecasting rather than description.